# Ariadne SQL Catalog

Access Ariadne indexes as Spark SQL tables. Every index under `spark.ariadne.storagePath`
is automatically discoverable — no registration needed.

**Important:** `spark.sql.extensions` must be set on `SparkConf` **before** the SparkContext
is created. The `%%init_spark` cell below handles this.

In [1]:
%%init_spark
launcher.master = "local[*]"
launcher.conf.set("spark.ariadne.storagePath", "/home/ariadne_storage")
launcher.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
launcher.conf.set("spark.sql.catalog.ariadne", "dev.cjfravel.ariadne.catalog.AriadneCatalog")
launcher.conf.set("spark.sql.extensions", "dev.cjfravel.ariadne.catalog.AriadneSparkExtension")

## Setup: Create Sample Indexes

First, create some indexes using the Scala API (the catalog is read-only).

In [2]:
import dev.cjfravel.ariadne.Index
import org.apache.spark.sql.types._
implicit val spark = org.apache.spark.sql.SparkSession.builder().getOrCreate()

// Clean up any previous index data
import org.apache.hadoop.fs.{FileSystem, Path}
val fs = FileSystem.get(spark.sparkContext.hadoopConfiguration)
fs.delete(new Path("/home/ariadne_storage"), true)

// Create an Ariadne index for customers (large dataset you query often)
val customerSchema = StructType(Seq(
  StructField("customer_id", IntegerType),
  StructField("name", StringType),
  StructField("email", StringType),
  StructField("region", StringType),
  StructField("signup_date", StringType)
))
val customers = Index("sql_customers", customerSchema, "csv", Map("header" -> "true"))
customers.addFile("/home/data/customers_west.csv")
customers.addFile("/home/data/customers_east.csv")
customers.addIndex("customer_id")
customers.addIndex("region")
customers.update

// Load orders as a regular Spark table (the ad-hoc query side of the join)
spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("/home/data/orders_q1.csv", "/home/data/orders_q2.csv")
  .createOrReplaceTempView("orders")

println("Setup complete!")


Intitializing Scala interpreter ...

Spark Web UI available at http://a1c023bd9c3e:4041
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1775582030465)
SparkSession available as 'spark'

Setup complete!


import dev.cjfravel.ariadne.Index
import org.apache.spark.sql.types._
spark: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@4cb5beee
import org.apache.hadoop.fs.{FileSystem, Path}
fs: org.apache.hadoop.fs.FileSystem = org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem@71ae2b6e
customerSchema: org.apache.spark.sql.types.StructType = StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))
customers: dev.cjfravel.ariadne.Index = Index(sql_customers,Some(StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup...

## Discovering Indexes

`SHOW TABLES IN ariadne` lists all indexes. `DESCRIBE` shows the schema.

In [3]:
spark.sql("SHOW TABLES IN ariadne").show(false)

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|default  |sql_customers|false      |
+---------+-------------+-----------+



In [4]:
spark.sql("DESCRIBE ariadne.sql_customers").show(false)

+-----------+---------+-------+
|col_name   |data_type|comment|
+-----------+---------+-------+
|customer_id|int      |NULL   |
|name       |string   |NULL   |
|email      |string   |NULL   |
|region     |string   |NULL   |
|signup_date|string   |NULL   |
+-----------+---------+-------+



## SELECT: Reading Data

`SELECT *` reads all source data files. Column projection is pushed down.

In [5]:
spark.sql("SELECT * FROM ariadne.sql_customers").show()

+-----------+--------------+-----------------+------+-----------+
|customer_id|          name|            email|region|signup_date|
+-----------+--------------+-----------------+------+-----------+
|       1001|   Alice Smith|alice@example.com|  West| 2024-01-15|
|       1002|   Bob Johnson|     bob@acme.com|  West| 2024-02-20|
|       1003|Carol Williams|carol@example.com|  West| 2024-03-10|
|       1004|     Grace Lee|grace@bigcorp.com|  West| 2024-06-01|
|       2001|    Dave Brown| dave@example.com|  East| 2024-01-05|
|       2002|     Eve Davis|   eve@startup.io|  East| 2024-04-01|
|       2003|  Frank Miller|frank@bigcorp.com|  East| 2024-05-15|
+-----------+--------------+-----------------+------+-----------+



In [6]:
// Column projection — only reads the requested columns
spark.sql("SELECT customer_id, name FROM ariadne.sql_customers").show()

+-----------+--------------+
|customer_id|          name|
+-----------+--------------+
|       1001|   Alice Smith|
|       1002|   Bob Johnson|
|       1003|Carol Williams|
|       1004|     Grace Lee|
|       2001|    Dave Brown|
|       2002|     Eve Davis|
|       2003|  Frank Miller|
+-----------+--------------+



## WHERE: Filter Pushdown

Equality filters on indexed columns use `locateFiles()` to prune the file list
before reading. Ariadne only reads files that contain matching values.

In [7]:
// This only reads the file(s) containing customer_id = 1001
spark.sql("SELECT * FROM ariadne.sql_customers WHERE customer_id = 1001").show()

+-----------+-----------+-----------------+------+-----------+
|customer_id|       name|            email|region|signup_date|
+-----------+-----------+-----------------+------+-----------+
|       1001|Alice Smith|alice@example.com|  West| 2024-01-15|
+-----------+-----------+-----------------+------+-----------+



In [8]:
// IN clause also uses file pruning
spark.sql("SELECT * FROM ariadne.sql_customers WHERE region IN ('East')").show()

+-----------+------------+-----------------+------+-----------+
|customer_id|        name|            email|region|signup_date|
+-----------+------------+-----------------+------+-----------+
|       2001|  Dave Brown| dave@example.com|  East| 2024-01-05|
|       2002|   Eve Davis|   eve@startup.io|  East| 2024-04-01|
|       2003|Frank Miller|frank@bigcorp.com|  East| 2024-05-15|
+-----------+------------+-----------------+------+-----------+



## JOIN: Automatic Optimization

When you JOIN against an Ariadne table, the **AriadneJoinRule** optimizer
automatically pre-prunes files using the join key values. This happens
transparently — you just write normal SQL.

The optimizer fires for **INNER equi-joins** where all join columns are indexed.

In [9]:
// JOIN: Ariadne prunes customer files to only those containing
// customer_ids present in the orders table — no full scan needed
spark.sql("""
  SELECT c.customer_id, c.name, c.region, o.order_id, o.amount
  FROM ariadne.sql_customers c
  JOIN orders o ON c.customer_id = o.customer_id
  ORDER BY o.order_id
""").show()


+-----------+--------------+------+--------+------+
|customer_id|          name|region|order_id|amount|
+-----------+--------------+------+--------+------+
|       1001|   Alice Smith|  West|   10001| 29.99|
|       2001|    Dave Brown|  East|   10002|149.99|
|       1002|   Bob Johnson|  West|   10003| 29.99|
|       1003|Carol Williams|  West|   10004|  79.5|
|       2002|     Eve Davis|  East|   10005| 29.99|
|       1001|   Alice Smith|  West|   10006|199.99|
|       2003|  Frank Miller|  East|   10007|149.99|
|       1001|   Alice Smith|  West|   10008|  79.5|
+-----------+--------------+------+--------+------+



## JOIN with Different Column Names

The optimizer supports joins where column names differ between sides.
It identifies which attribute belongs to the Ariadne side by ExprId.

In [10]:
import org.apache.spark.sql.Row

// Create a lookup table with a differently-named column
val lookupDf = spark.createDataFrame(
  spark.sparkContext.parallelize(Seq(Row(1001), Row(2002))),
  StructType(Seq(StructField("cust_id", IntegerType)))
)
lookupDf.createOrReplaceTempView("lookup")

// Join on c.customer_id = l.cust_id (different names)
spark.sql("""
  SELECT c.customer_id, c.name, c.region
  FROM ariadne.sql_customers c
  JOIN lookup l ON c.customer_id = l.cust_id
""").show()

+-----------+-----------+------+
|customer_id|       name|region|
+-----------+-----------+------+
|       1001|Alice Smith|  West|
|       2002|  Eve Davis|  East|
+-----------+-----------+------+



import org.apache.spark.sql.Row
lookupDf: org.apache.spark.sql.DataFrame = [cust_id: int]

## LEFT JOIN Fallback

Non-inner joins (LEFT, RIGHT, FULL) fall back to the V1Scan path (full table scan).
They still produce correct results — just without file pruning optimization.

In [11]:
val missingDf = spark.createDataFrame(
  spark.sparkContext.parallelize(Seq(Row(1001), Row(9999))),
  StructType(Seq(StructField("customer_id", IntegerType)))
)
missingDf.createOrReplaceTempView("maybe_customers")

// LEFT JOIN — unmatched rows get nulls for the Ariadne side
spark.sql("""
  SELECT m.customer_id, c.name, c.region
  FROM maybe_customers m
  LEFT JOIN ariadne.sql_customers c ON m.customer_id = c.customer_id
""").show()

+-----------+-----------+------+
|customer_id|       name|region|
+-----------+-----------+------+
|       1001|Alice Smith|  West|
|       9999|       NULL|  NULL|
+-----------+-----------+------+



missingDf: org.apache.spark.sql.DataFrame = [customer_id: int]

## JOIN + WHERE Combined

You can combine JOINs with WHERE filters. The optimizer handles the join
while Spark applies the WHERE filter post-join.

In [12]:
// Combine JOIN with additional filters
spark.sql("""
  SELECT c.customer_id, c.name, o.order_id, o.amount
  FROM ariadne.sql_customers c
  JOIN orders o ON c.customer_id = o.customer_id
  WHERE o.amount > 100
  ORDER BY o.amount DESC
""").show()


+-----------+------------+--------+------+
|customer_id|        name|order_id|amount|
+-----------+------------+--------+------+
|       1001| Alice Smith|   10006|199.99|
|       2001|  Dave Brown|   10002|149.99|
|       2003|Frank Miller|   10007|149.99|
+-----------+------------+--------+------+



## Optimization Summary

| Query Pattern | What Happens |
|--------------|-------------|
| `SELECT *` | Full scan of all source files |
| `WHERE col = val` | `locateFiles` prunes files before reading |
| `WHERE col IN (...)` | `locateFiles` prunes files before reading |
| `INNER JOIN ON indexed cols` | AriadneJoinRule pre-prunes files via distributed index lookup |
| `LEFT/RIGHT/FULL JOIN` | Falls back to full scan (correct results, no pruning) |
| Non-equi conditions | Falls back to full scan |

Index creation and updates are done through the **Scala API** — the SQL catalog is read-only.